# RSA and Rabin Functions: Certain Parts are as Hard as the Whole (Alexi et al., 1988) LeetArxiv Implementation

This coding guide is best followed alongside this [LeetArxiv article link](https://leetarxiv.substack.com/p/acgs-algorithm-for-hidden-number).



The paper, *RSA and Rabin Functions: Certain Parts are as Hard as the Whole* , introduces the **Alexi-Chor-Goldreich-Schnorr**(ACGS) algorithm for polynomial-time solutions to Hidden Number Problems with Chosen Multipliers (HNP-CM).

This is part of our series on Practical Hidden Number Problems for Programmers:

[Part 1: Quantum Computing and Intro to Hidden Number Problems
](https://leetarxiv.substack.com/p/linear-hidden-number-problem-zero-to-hero-for-computer-scientiests)

[Part 2: Increasing Lattice Volume for Improved HNP Attacks](https://leetarxiv.substack.com/p/guessing-bits-improved-lattice-attacks)

[Part 3: Hidden Number Problems with Multiple Noise Holes
](https://leetarxiv.substack.com/p/pythonsage-extended-hidden-number)

Part 4(We are here): [Hidden Number Problems with Chosen Multipliers](https://leetarxiv.substack.com/p/acgs-algorithm-for-hidden-number)

The paper is considered canonical and is included in MIT’s Foundations of Cryptography series. It introduces ACGS, an alternative to lattice construction techniques for solving HNPs.

ACGS splits into two parts:
1. Guessing the LSB at certain points.
2. Using interval reduction and the guessed LSB's to recover the secret key.

We go over the entire codebase in this [LeetArxiv article](https://leetarxiv.substack.com/p/acgs-algorithm-for-hidden-number).

In [ ]:
#Knowing a few MSBs and the LSB permits us guess correctly
import random
import math
from random import randint
def lsb(x): return x & 1

def GenerateRList(u, v, m, p):
    r = []
    for i in range(1, m+1):
        r_i = (i * u + v) % p
        r.append(r_i)
    return r

def RecoverLSB(i, yApproximate, zApproximate, yLSB, zLSB, eps, p):
    """
    Recover LSB([w_i]_p) using approximate y,z and true LSBs
    """
    # approximate w
    wApproximate = yApproximate + i * zApproximate
    # uncertainty interval
    interval = eps * p / 2
    lower = wApproximate - interval/2
    upper = wApproximate + interval/2
    k_low = math.floor(lower / p)
    k_high = math.floor(upper / p)
    ambiguous = (k_low != k_high)
    if ambiguous:
        q = k_high   # arbitrary choice from paper
    else:
        q = k_low
    # compute LSB(w_i)
    lsb_w = (yLSB + i * zLSB) % 2
    # compute LSB([w_i]_p)
    lsb_mod = lsb_w ^ (q & 1)
    return lsb_mod

def TestGuessLSB():
    prime = 205115282021455665897114700593932402728804164701536103180137503955397371
    epsilon = 0.3
    alpha = random.randint(1, prime-1)
    n = prime.bit_length();m = int(n * (1 / (epsilon ** 2)))
    knownMSBBitlength = math.ceil(2 * math.log(m, 2))
    #Random u,v
    u = randint(0, prime);v = randint(0, prime)
    r = GenerateRList(u,v,m,prime)
    trueY = (v * alpha) % prime
    trueZ = (u * alpha) % prime
    yLSB = trueY & 1
    zLSB = trueZ & 1
    yApproximate = (trueY >> knownMSBBitlength) << knownMSBBitlength
    zApproximate = (trueZ >> knownMSBBitlength) << knownMSBBitlength
    matchCount = 0
    for i in range(0, len(r)):
        #remember i starts at 1 when generating r
        true_lsb = RecoverLSB(i+1, yApproximate, zApproximate, yLSB, zLSB, epsilon, prime)
        targetLSB = lsb((r[i]*alpha)%prime)
        if(true_lsb == targetLSB):
            matchCount += 1
        print(f"{true_lsb}: {targetLSB} : {matchCount} / {len(r)} matches")

TestGuessLSB()

1: 1 : 1 / 2633 matches
0: 0 : 2 / 2633 matches
1: 1 : 3 / 2633 matches
0: 0 : 4 / 2633 matches
1: 1 : 5 / 2633 matches
0: 1 : 5 / 2633 matches
0: 0 : 6 / 2633 matches
1: 1 : 7 / 2633 matches
0: 0 : 8 / 2633 matches
1: 1 : 9 / 2633 matches
0: 0 : 10 / 2633 matches
1: 1 : 11 / 2633 matches
0: 0 : 12 / 2633 matches
0: 0 : 13 / 2633 matches
1: 1 : 14 / 2633 matches
0: 0 : 15 / 2633 matches
1: 1 : 16 / 2633 matches
0: 0 : 17 / 2633 matches
1: 1 : 18 / 2633 matches
0: 0 : 19 / 2633 matches
1: 0 : 19 / 2633 matches
1: 1 : 20 / 2633 matches
0: 0 : 21 / 2633 matches
1: 1 : 22 / 2633 matches
0: 0 : 23 / 2633 matches
1: 1 : 24 / 2633 matches
0: 0 : 25 / 2633 matches
1: 1 : 26 / 2633 matches
0: 1 : 26 / 2633 matches
0: 0 : 27 / 2633 matches
1: 1 : 28 / 2633 matches
0: 0 : 29 / 2633 matches
1: 1 : 30 / 2633 matches
0: 0 : 31 / 2633 matches
1: 1 : 32 / 2633 matches
0: 0 : 33 / 2633 matches
0: 0 : 34 / 2633 matches
1: 1 : 35 / 2633 matches
0: 0 : 36 / 2633 matches
1: 1 : 37 / 2633 matches
0: 0 : 38 

In [ ]:
#Evaluating our HNP at multiples of 2 using guessed bits from before
import random

p = 13441
alpha = random.randrange(1, p)

def oracle(t):
    return ((alpha * t) % p) & 1

def RecoverAlpha_IntervalReduction():
    low = 0
    high = p
    t = 1
    for _ in range(p.bit_length() + 2):
        bit = oracle(t * 2)
        mid = (low + high) // 2
        if bit == 0:
            # no reduction → alpha*t mod p < p/2
            high = mid
        else:
            # reduction happened → alpha*t mod p >= p/2
            low = mid
        t *= 2
    return low

guess = RecoverAlpha_IntervalReduction()

print("real alpha:", alpha)
print("recovered :", guess)

real alpha: 8593
recovered : 8590
